# Judge validation — does the LLM correctness judge agree with a human? — corrected

Every correctness figure in the project comes from an LLM deciding whether an answer matches the
gold. Nobody has checked that judge against a person, so the generation half currently rests on an
unvalidated instrument.

**Two passes, with your labelling in between:**

1. Run the cells up to "Download the sheet" → produces `labeling_sheet.csv`
2. Fill `human_correct` with 1 or 0 for each row (~15–20 min for 100 short factoid answers)
3. Run the remaining cells → Cohen's κ

**Don't look at `llm_correct` while labelling.** Seeing the judge's answer first biases your label
and inflates agreement.

## There is no TPU version of this notebook, and there should not be

This notebook does no tensor computation at all. It reads CSVs, samples rows, and computes Cohen's
κ with scikit-learn — all of it on a handful of hundreds of rows, in well under a second. There is
nothing here for an accelerator to do, so a "TPU edition" would differ from this file only by a
metadata string claiming hardware it never touches. That would be misleading rather than useful, so
it was not produced. The original's own header already said **"No GPU needed"** — that is correct
and unchanged.

## Defects fixed

**`generation_n200_raw.csv` does not exist.** It is not in the repo, not in `results/`, and not in
git history. The loop `if not os.path.exists(path): print("skipping")` swallowed it, so the sheet
silently came out at 50 rows instead of the 100 the header promises — and a κ computed on one run
was presented as validating both. The config now names the file that exists
(`generation_reranked_raw.csv` and `generation_local_raw.csv`), resolves each through the repo's
`data/` and `results/` directories, and **prints a warning for every CSV it could not find** rather
than passing over it quietly.

**`stratified_sample(df, 50, …)` returns 48 rows, not 50.** `n_per = 50 // 4 = 12`, twelve per
condition, four conditions. Now topped up to the requested size.

**The final cell could raise `NameError`.** `files.download(...)` at the end of the scoring cell
relies on `from google.colab import files` having run in an *earlier* cell. Run the scoring cells
alone — which is exactly what the two-pass workflow tells you to do — and that name is undefined.
Both download calls are now guarded independently.

**Cohen's κ is undefined when one rater uses a single class.** If every `human_correct` is 1 (quite
possible on a 50-row sample of a system scoring ~80%), `cohen_kappa_score` returns `nan` and the
interpretation band silently reports "slight — the judge is not usable", which is wrong: it is
undefined, not bad. Now detected and explained.

**`human_correct` accepted any integer.** The check ran `astype(int)` before validating, so a stray
`2` or `-1` became a silent miscount. Values are now validated before conversion, and the offending
rows are named.

### Install

In [ ]:
import importlib.util, subprocess, sys

need = [p for p, m in [("pandas", "pandas"), ("scikit-learn", "sklearn")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=False)

print("deps ready")

### Paths

In [ ]:
import os, json, re, gc, time, random, pickle, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

RAW_BASE = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data"

def _repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "qa_pairs_wiki.json").exists():
            return base
    return None

ROOT = _repo_root()

def find_file(*names, subdirs=("data", "results")):
    """Locate a file: repo subdirs first, then the working directory."""
    for n in names:
        if ROOT:
            for sd in subdirs:
                p = ROOT / sd / n
                if p.exists():
                    return p
        if Path(n).exists():
            return Path(n)
    return None

def resolve(*names):
    """Local -> GitHub. Returns (json, description)."""
    p = find_file(*names, subdirs=("data",))
    if p:
        return json.loads(p.read_text(encoding="utf-8")), f"local: {p}"
    for n in names:
        try:
            url = f"{RAW_BASE}/{n}"
            with urllib.request.urlopen(url) as r:
                return json.loads(r.read().decode("utf-8")), f"github: {url}"
        except Exception:
            continue
    raise FileNotFoundError(f"none of {names} found locally or on GitHub")

OUT_DIR = (ROOT / "results") if ROOT else Path(".")
OUT_DIR.mkdir(exist_ok=True)
def out(name):
    return str(OUT_DIR / name)

print("repo root :", ROOT or "(not in the repo)")
print("output dir:", OUT_DIR.resolve())

### Config

In [ ]:
CONFIG = {
    # Generation CSVs to validate. The original listed "generation_n200_raw.csv",
    # which exists nowhere in this repo -- it was skipped silently, halving the sheet.
    "generation_csvs": [
        "generation_reranked_raw.csv",   # the n=200 reranked run
        "generation_local_raw.csv",      # the earlier run (this is the file that exists)
    ],
    "n_per_file": 50,          # sampled per CSV
    "sheet_out": "labeling_sheet.csv",
    "scored_out": "labeling_sheet_scored.csv",
    "seed": 42,
}
CONFIG

### Build the labelling sheet

In [ ]:
wiki_qa, src_q = resolve("qa_pairs_wiki.json")
qa_lookup = {q["id"]: q for q in wiki_qa}
print("qa from", src_q, f"({len(qa_lookup)} items)")

def stratified_sample(df, n, run_label):
    """Balanced across conditions AND across the judge's own verdict.

    Deliberately oversamples what the judge called WRONG: a purely random sample
    is dominated by obviously-correct answers where agreement is trivial, and
    disagreements concentrate in the hard cases.
    """
    conds = max(df["condition"].nunique(), 1)
    n_per = max(1, n // conds)
    parts = []
    for cond, grp in df.groupby("condition"):
        wrong, right = grp[grp["correct"] == 0], grp[grp["correct"] == 1]
        n_wrong = min(len(wrong), max(1, n_per // 2))
        n_right = min(len(right), n_per - n_wrong)
        parts.append(pd.concat([
            wrong.sample(n_wrong, random_state=CONFIG["seed"]) if n_wrong else wrong.head(0),
            right.sample(n_right, random_state=CONFIG["seed"]) if n_right else right.head(0),
        ]))
    sample = pd.concat(parts)

    # n // conds * conds < n unless n divides evenly: 50 // 4 * 4 = 48.
    # Top up so the sheet is the size asked for.
    if len(sample) < n:
        rest = df.drop(index=sample.index, errors="ignore")
        if len(rest):
            sample = pd.concat([sample, rest.sample(min(n - len(sample), len(rest)),
                                                    random_state=CONFIG["seed"])])
    sample = sample.sample(frac=1, random_state=CONFIG["seed"]).reset_index(drop=True).head(n)

    rows, missing = [], 0
    for _, r in sample.iterrows():
        q = qa_lookup.get(r["qid"])
        if not q:
            missing += 1          # would otherwise be an all-blank, unlabelable row
            continue
        rows.append({
            "run": run_label, "qid": r["qid"], "condition": r["condition"],
            "msa_query": q.get("msa_query", ""), "darija_query": q.get("darija_query", ""),
            "gold_answer": q.get("gold_answer", ""), "model_answer": r.get("answer", ""),
            "llm_correct": int(r["correct"]), "human_correct": "",
        })
    if missing:
        print(f"    WARNING: {missing} rows dropped - qid not in the benchmark.")
    return pd.DataFrame(rows)

sheets, missing_files = [], []
for name in CONFIG["generation_csvs"]:
    path = find_file(name)
    if path is None:
        missing_files.append(name)
        print(f"  NOT FOUND: {name}")
        continue
    df = pd.read_csv(path)
    if "correct" not in df.columns:
        print(f"  SKIPPED (no 'correct' column): {path}")
        continue
    label = Path(name).stem
    sheets.append(stratified_sample(df, CONFIG["n_per_file"], label))
    print(f"  sampled {CONFIG['n_per_file']} from {path}  ({len(df)} rows, "
          f"conditions: {', '.join(sorted(df.condition.unique()))})")

if not sheets:
    raise RuntimeError("No usable generation CSVs found - run a generation notebook first.")

if missing_files:
    print(f"\n  WARNING: {len(missing_files)} of {len(CONFIG['generation_csvs'])} CSVs were not "
          f"found ({', '.join(missing_files)}).")
    print("  The sheet below covers only the runs listed above, so kappa validates")
    print("  those runs only -- not every run in the project.")

combined = pd.concat(sheets, ignore_index=True)
sheet_path = out(CONFIG["sheet_out"])
combined.to_csv(sheet_path, index=False, encoding="utf-8-sig")   # utf-8-sig: Excel shows Arabic
print(f"\nWrote {len(combined)} rows to {sheet_path}")
print(combined.groupby("run").size().to_string())
print(f"\nLLM said correct: {combined['llm_correct'].sum()}/{len(combined)}")

### Download the sheet, then label it

In [ ]:
try:
    from google.colab import files as colab_files
    colab_files.download(out(CONFIG["sheet_out"]))
except ImportError:
    print(f"(Not in Colab - the sheet is at {out(CONFIG['sheet_out'])})")

print("""
NEXT STEPS
  1. Open labeling_sheet.csv in Excel or Google Sheets.
  2. For each row, read 'gold_answer' against 'model_answer'.
  3. Put 1 in 'human_correct' if the model's answer is correct, 0 if not.
     Wording may differ freely; numbers, names and dates must match.
     Do NOT look at 'llm_correct' while labelling -- seeing the judge's answer
     first biases the human label and inflates agreement.
  4. Save as CSV under the same name, then run the remaining cells.
""")

### Load the filled sheet

In [ ]:
SHEET = find_file(CONFIG["sheet_out"]) or out(CONFIG["sheet_out"])
print("reading", SHEET)
df = pd.read_csv(SHEET)

raw = df["human_correct"]
unfilled = raw.isna() | raw.astype(str).str.strip().eq("")
if unfilled.any():
    raise ValueError(f"{unfilled.sum()} row(s) unlabelled "
                     f"(qid: {df.loc[unfilled, 'qid'].tolist()[:10]}...). Fill them all first.")

# Validate BEFORE converting: the original ran astype(int) first, so a stray 2
# or -1 became a silent miscount rather than an error.
bad = raw[~raw.astype(str).str.strip().isin(["0", "1", "0.0", "1.0"])]
if len(bad):
    raise ValueError(f"'human_correct' must be 0 or 1. Offending rows - "
                     f"{df.loc[bad.index, 'qid'].tolist()[:10]} (values {bad.tolist()[:10]})")

human = raw.astype(float).astype(int)
llm = df["llm_correct"].astype(int)
print(f"Loaded {len(df)} labelled rows.")

### Cohen's kappa

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix, accuracy_score

acc = accuracy_score(human, llm)
cm = confusion_matrix(human, llm, labels=[0, 1])

print("=" * 62); print("JUDGE VALIDATION"); print("=" * 62)
print(f"n = {len(df)}")
print(f"Raw agreement:  {acc:.3f}")

# Kappa is undefined when either rater uses only one class: the expected-agreement
# denominator collapses. The original reported the resulting nan as "slight -- the
# judge is not usable", which is a wrong conclusion from an undefined statistic.
single_class = human.nunique() < 2 or llm.nunique() < 2
if single_class:
    kappa = float("nan")
    print("Cohen's kappa:  UNDEFINED")
    print(f"  (human used {human.nunique()} class(es), LLM used {llm.nunique()})")
    print("  Kappa needs both raters to use both labels. This is not a bad kappa --")
    print("  it is no kappa. Report raw agreement, and sample more disagreement cases.")
else:
    kappa = cohen_kappa_score(human, llm)
    print(f"Cohen's kappa:  {kappa:.3f}")

print()
print("Confusion matrix (rows = human, cols = LLM judge):")
print("              LLM=0    LLM=1")
print(f"  human=0     {cm[0,0]:>5}    {cm[0,1]:>5}")
print(f"  human=1     {cm[1,0]:>5}    {cm[1,1]:>5}")

band = "undefined"
if not single_class:
    # Landis & Koch (1977) bands
    if kappa < 0.20:   band = "slight - the judge is not usable as reported"
    elif kappa < 0.40: band = "fair - weak; revise the judge prompt"
    elif kappa < 0.60: band = "moderate - borderline; report with explicit caution"
    elif kappa < 0.80: band = "substantial - defensible for reporting"
    else:              band = "almost perfect - strong"
    print(f"\nInterpretation: {band}")

print(f"\nClass balance (human labels): {human.mean():.1%}")
if human.mean() > 0.85 or human.mean() < 0.15:
    print("  Note: with a heavily skewed class balance kappa is pessimistic.")
    print("  Report raw agreement alongside it.")

### Inspect disagreements, and per-run breakdown

In [ ]:
dis = df[human.values != llm.values]
print("\n" + "=" * 62); print(f"DISAGREEMENTS ({len(dis)} of {len(df)})"); print("=" * 62)
if len(dis):
    for _, r in dis.iterrows():
        print(f"\n[{r['run']} | {r['condition']}]")
        print(f"  gold:  {r['gold_answer']}")
        print(f"  model: {str(r['model_answer'])[:200]}")
        print(f"  llm={r['llm_correct']}  human={r['human_correct']}")
else:
    print("None - perfect agreement.")

if df["run"].nunique() > 1:
    print("\n" + "=" * 62); print("PER-RUN AGREEMENT"); print("=" * 62)
    for run, grp in df.groupby("run"):
        h = grp["human_correct"].astype(float).astype(int)
        l = grp["llm_correct"].astype(int)
        k = (cohen_kappa_score(h, l) if h.nunique() > 1 and l.nunique() > 1 else float("nan"))
        ks = f"{k:.3f}" if k == k else "undefined"
        print(f"  {run:<32} n={len(grp):<4} kappa={ks}  agreement={accuracy_score(h, l):.3f}")
    print("\n  Similar kappa across runs means the judge behaves consistently")
    print("  regardless of the retrieval setup being evaluated.")

scored = out(CONFIG["scored_out"])
df.to_csv(scored, index=False, encoding="utf-8-sig")
print(f"\nWrote {scored}")

ktxt = f"{kappa:.2f}" if kappa == kappa else "undefined"
print(f"""
FOR THE PAPER
  "Correctness labels were assigned by an LLM judge. On a stratified sample of
   {len(df)} generated answers spanning {df['run'].nunique()} run(s), LLM and human
   labels agreed with Cohen's kappa = {ktxt} (raw agreement {acc:.1%}), indicating
   {band.split(' - ')[0]} agreement."
""")

# Guarded independently: the original called files.download() here relying on an
# import from an earlier cell, so running the scoring pass alone raised NameError.
try:
    from google.colab import files as colab_files
    colab_files.download(scored)
except ImportError:
    print(f"(Not in Colab - scored sheet is at {scored})")